In [1]:
import duckdb
import pandas as pd
import numpy as np
import geopandas as gpd
import shapely
import contextily as ctx
from shapely import wkt
import time
import random
import json
import folium
from shapely.geometry import mapping
from shapely.geometry import mapping
from branca.colormap import linear
from shapely.geometry import LineString, MultiLineString, Point, MultiPoint, GeometryCollection
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
import os

In [2]:
print(duckdb.__version__)
# The MobilityDuck CI builds and tests the extension against DuckDB v1.5.6, and
# the extension loads only into the DuckDB version it was built for.
assert duckdb.__version__ == '1.5.6'
# The extension built by the MobilityDuck CI, or by a build of MobilityDuck
MOBILITYDUCK_EXTENSION = os.environ.get('MOBILITYDUCK_EXTENSION',
  '/path/to/mobilityduck.duckdb_extension')
# The PostgreSQL database of the Hanoi data, which load/hanoi_pg.sql loads
PG_HANOI = os.environ.get('PG_HANOI',
  'host=localhost port=55432 dbname=Hanoi user=postgres password=mysecretpassword')

1.4.3


In [3]:
connection = duckdb.connect(
    config={'allow_unsigned_extensions' : 'true'}
)

In [4]:
connection = duckdb.connect(
    database='hanoi.db',
    config={'allow_unsigned_extensions' : 'true'}
)

In [5]:
connection.execute("INSTALL spatial")

In [6]:
connection.execute(f"INSTALL '{MOBILITYDUCK_EXTENSION}'")

In [7]:
connection.load_extension(MOBILITYDUCK_EXTENSION)
connection.load_extension('spatial')

# Create tables and load data

In [8]:
connection.execute("SET threads=1");
# The timestamps of the BerlinMOD-Hanoi data set are in UTC
connection.execute("SET TimeZone='UTC'");

In [9]:
connection.execute("SET memory_limit='7000MB'");

In [10]:
connection.execute("SET preserve_insertion_order=false");

In [11]:
connection.execute("SET temp_directory = 'hanoi.tmp/'");

In [17]:
query = """
CREATE OR REPLACE TABLE trips (
    tripid integer PRIMARY KEY,
    vehicleid integer NOT NULL,
    trip tgeompoint,
    trajectory geometry
);
"""
connection.execute(query)

In [18]:
conn_str = PG_HANOI

query = f"""
INSERT INTO trips
SELECT tripid,
     vehicleid,
     trip tgeompoint,
     trajectory   FROM postgres_scan(
    '{conn_str}',    
    'public',          
    'trips'       
);
"""
result = connection.execute(query).fetchone()

In [7]:
connection.execute("""
SELECT tripid, vehicleid, astext(trip), st_astext(trajectory) FROM Trips Limit 3
""").fetchdf()

,tripid,vehicleid,astext(trip),st_astext(trajectory)
0,1,1,[POINT(11781545.407382663 2393706.095838943)@2...,LINESTRING (11781545.407382663 2393706.0958389...
1,2,1,[POINT(11733388.929623965 2421155.8332119286)@...,LINESTRING (11733388.929623965 2421155.8332119...
2,3,1,[POINT(11781545.407382663 2393706.095838943)@2...,LINESTRING (11781545.407382663 2393706.0958389...


In [26]:
query = """SELECT COUNT(*) FROM trips"""
result = connection.execute(query).fetchdf()
result

,count_star()
0,2903


In [ ]:
query = """
CREATE OR REPLACE TABLE municipalities(
    municipalityid BIGINT PRIMARY KEY,
    municipalityname VARCHAR,
    population BIGINT,
    percpop DOUBLE,
    popdensitykm2 BIGINT,
    noenterp BIGINT,
    percenterp DOUBLE,
    municipalitygeo GEOMETRY
);
"""
connection.execute(query)

In [ ]:
conn_str = PG_HANOI

query = f"""
INSERT INTO municipalities
SELECT *  FROM postgres_scan(
    '{conn_str}',    
    'public',          
    'municipalities'       
);
"""
result = connection.execute(query).fetchone()

In [ ]:
conn_str = PG_HANOI

query = f"""
INSERT INTO municipalities
SELECT municipalityid,
    municipalityname,
    population,
    percpop,
    popdensitykm2,
    noenterp,
    percenterp,
    municipalitygeo FROM postgres_scan(
    '{conn_str}',    
    'public',          
    'municipalities'       
);
"""
result = connection.execute(query).fetchone()

In [34]:
query = """SELECT municipalityid, municipalityname, population,
    percpop, popdensitykm2, noenterp, percenterp,
    st_astext(municipalitygeo) FROM municipalities Limit 2"""
result = connection.execute(query).fetchdf()
result

,municipalityid,municipalityname,population,percpop,popdensitykm2,noenterp,percenterp,st_astext(municipalitygeo)
0,30,Huyện Ứng Hòa,215900,0.03,1147,3852,0.03,"POLYGON ((11766093.96149793 2361530.927014738,..."
1,21,Huyện Mỹ Đức,210200,0.02,929,3751,0.02,POLYGON ((11760645.796375375 2365247.918583854...


In [ ]:
query = """
CREATE OR REPLACE TABLE Instants(
    instantId integer,
    instant TIMESTAMPTZ
);
"""
connection.execute(query)

In [ ]:
conn_str = PG_HANOI

query = f"""
INSERT INTO Instants
SELECT * FROM postgres_scan(
    '{conn_str}',    
    'public',          
    'instants'      
);
"""
result = connection.execute(query).fetchone()

In [23]:
query = """SELECT * FROM Instants"""
result = connection.execute(query).fetchdf()
result

,instantId,instant
0,1,2020-06-05 00:21:07.312143-03:00
1,2,2020-06-02 14:50:50.130383-03:00
2,3,2020-06-02 10:16:56.912297-03:00
3,4,2020-06-02 21:47:33.393991-03:00
4,5,2020-06-04 15:48:17.206196-03:00
...,...,...
95,96,2020-06-04 14:19:39.224466-03:00
96,97,2020-06-03 00:16:23.037095-03:00
97,98,2020-06-02 11:41:19.915152-03:00
98,99,2020-06-01 04:51:09.005105-03:00


In [ ]:
query = """
CREATE OR REPLACE TABLE Periods(
    PeriodId integer,
    Period TIMESTAMPTZ
);
"""
connection.execute(query)

In [ ]:
query = "SELECT ST_AsText(trajectory) as traj  FROM trips;"
trajs = connection.execute(query).df()
trajs['geom_obj'] = trajs['traj'].apply(lambda x: shapely.wkt.loads(x))
trajs_gdf = gpd.GeoDataFrame(trajs, geometry='geom_obj')
trajs_gdf['color'] = 'crimson'
trajs_gdf.head(2)

,traj,geom_obj,color
0,LINESTRING (11781545.407382663 2393706.0958389...,"LINESTRING (11781545.407 2393706.096, 11781550...",crimson
1,LINESTRING (11733388.929623965 2421155.8332119...,"LINESTRING (11733388.93 2421155.833, 11733388....",crimson


In [44]:
query1 = "SELECT st_astext(municipalitygeo) as muni  FROM municipalities;"
hanoi = connection.execute(query1).df()
hanoi.head()

,muni
0,"POLYGON ((11766093.96149793 2361530.927014738,..."
1,POLYGON ((11760645.796375375 2365247.918583854...
2,POLYGON ((11741779.814353693 2385764.425382919...
3,POLYGON ((11749886.032881565 2368990.853617009...
4,POLYGON ((11776689.484215025 2363453.482782691...


In [47]:
hanoi['geom_obj'] = hanoi['muni'].apply(lambda x: shapely.wkt.loads(x))

In [48]:
hanoi_gdf = gpd.GeoDataFrame(hanoi, geometry='geom_obj')
hanoi_gdf['color'] = sns.color_palette('hls', hanoi_gdf.shape[0]).as_hex()
hanoi_gdf.head(2)

,muni,geom_obj,color
0,"POLYGON ((11766093.96149793 2361530.927014738,...","POLYGON ((11766093.961 2361530.927, 11766147.6...",#db5f57
1,POLYGON ((11760645.796375375 2365247.918583854...,"POLYGON ((11760645.796 2365247.919, 11760834.0...",#db7957


In [ ]:
# District in Hanoi and trips
_, axes = plt.subplots(figsize=(5, 5), dpi=150)
hanoi_gdf.plot(color=hanoi_gdf["color"], ax=axes, alpha=0.6)
trajs_gdf.plot(color=trajs_gdf["color"], ax=axes, linewidth=0.3)

axes.set_title("Districts of Hanoi")

plt.show()

In [ ]:
query = """
WITH tripInt AS
 (SELECT vehicleid, tripid,
         COUNT(DISTINCT m.municipalityid) AS intersectCount
  FROM trips t, municipalities m
  WHERE eintersects(trip, municipalitygeo)
GROUP BY vehicleid, tripid)
SELECT t1.*, ST_AsText(t.trajectory) AS trajectory
FROM trips t JOIN tripInt t1 ON (t.tripId = t1.tripId AND t.vehicleid = t1.vehicleid)
ORDER BY intersectCount DESC;
"""

intersect_count_df = connection.execute(query).df()
intersect_count_df['geom_obj'] = intersect_count_df['trajectory'].apply(lambda x: shapely.wkt.loads(x))
intersect_count_gdf = gpd.GeoDataFrame(intersect_count_df, geometry='geom_obj')
intersect_count_gdf

In [76]:
# # Query 6.26. Trips that cross the largest number of districts.
query = """
   WITH tripInt AS
 (SELECT vehicleid, tripid,
         COUNT(DISTINCT m.municipalityid) AS intersectCount
  FROM trips t, municipalities m
  WHERE eintersects(trip, municipalitygeo)
GROUP BY vehicleid, tripid)
SELECT t1.*, ST_AsText(t.trajectory) AS trajectory
FROM trips t JOIN tripInt t1 ON (t.tripId = t1.tripId AND t.vehicleid = t1.vehicleid)
ORDER BY intersectCount DESC
LIMIT 1;"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

Execution time: 28.313549 seconds


,vehicleid,tripid,intersectCount,trajectory
0,8,105,15,LINESTRING (11733208.057715323 2421402.5553750...


In [ ]:
query = """
WITH tripInt AS
 (SELECT vehicleid, tripid,
         COUNT(DISTINCT m.municipalityid) AS intersectCount
  FROM trips t, municipalities m
  WHERE eintersects(trip, municipalitygeo)
GROUP BY vehicleid, tripid)
SELECT t1.*, ST_AsText(t.trajectory) AS trajectory
FROM trips t JOIN tripInt t1 ON (t.tripId = t1.tripId AND t.vehicleid = t1.vehicleid)
ORDER BY intersectCount DESC
LIMIT 1;
"""
intersect_count_df = connection.execute(query).df()
intersect_count_df['geom_obj'] = intersect_count_df['trajectory'].apply(lambda x: shapely.wkt.loads(x))
intersect_count_gdf = gpd.GeoDataFrame(intersect_count_df, geometry='geom_obj')
intersect_count_gdf

,vehicleid,tripid,intersectCount,trajectory,geom_obj
0,8,105,15,LINESTRING (11733208.057715323 2421402.5553750...,"LINESTRING (11733208.058 2421402.555, 11733210..."


In [ ]:
_, axes = plt.subplots(figsize=(10, 10), dpi=150)
hanoi_gdf.plot(color=hanoi_gdf["color"], ax=axes, alpha=0.6)

axes.set_title("Districts of Hanoi")
trips_plot = intersect_count_gdf 
#trips_plot = intersect_count_gdf[intersect_count_gdf['intersect_count'] == intersect_count_gdf['intersect_count'].max()]
trips_plot['color'] = 'crimson'
trips_plot.plot(ax=axes, color=trips_plot['color'], linewidth=1)

plt.show()

In [ ]:
# Query 6.26. Fig. 6.22
# The districts and the trip in longitude and latitude
hanoi_gdf_4326 = hanoi_gdf.set_crs(3857).to_crs(4326)
trip_4326 = intersect_count_gdf.set_crs(3857).to_crs(4326).geometry.iloc[0]
# Map center
center = hanoi_gdf_4326.geometry.union_all().centroid
m = folium.Map(location=[center.y, center.x], zoom_start=11, tiles="CartoDB positron")

# Add districts: the districts the trip crosses take, in the order the trip enters them, the
# colours of tab10 but its red and grey, and the others are grey, each bounded by a white line,
# so that the crossings can be counted on the map
palette = [c for i, c in enumerate(plt.colormaps['tab10'].colors) if i not in (3, 7)]
crossed = [i for i in hanoi_gdf_4326.index if hanoi_gdf_4326.geometry[i].intersects(trip_4326)]
crossed.sort(key=lambda i: trip_4326.project(
    hanoi_gdf_4326.geometry[i].intersection(trip_4326).representative_point()))
colors = {i: mcolors.to_hex(palette[k % len(palette)]) for k, i in enumerate(crossed)}
for i, row in hanoi_gdf_4326.iterrows():
    folium.GeoJson(
        mapping(row["geom_obj"]),  # use the geometry column directly
        style_function=lambda feature, col=colors.get(i, "#b4b4b4"): {
            "fillColor": col,
            "color": "white",
            "weight": 1.2,
            "fillOpacity": 0.6,
        }
    ).add_to(m)

# Add the trip
folium.GeoJson(
    mapping(trip_4326),
    style_function=lambda feature: {
        "color": "crimson",
        "weight": 2,
        "opacity": 1
    },
    tooltip=f"intersectCount: {intersect_count_gdf['intersectCount'].iloc[0]}"
).add_to(m)
m

In [ ]:
# Query 6.27 Trips crossing each district

def get_trip_colors(n_colors):
    og_colors = list(mcolors.TABLEAU_COLORS.values())
    while len(og_colors) < n_colors:
        og_colors = og_colors * 2
    return random.sample(og_colors, n_colors)
    
def get_district_trips(municipalityName):
    query = f"""
SELECT  st_astext(trajectory(atGeometry(t.trip, m.municipalitygeo)))  as trajClip,
FROM trips t, municipalities m
WHERE eintersects(t.trip, m.municipalitygeo) 
    AND m.municipalityname = '{municipalityName}';
"""
    df = connection.execute(query).df()
    df['geom'] = df['trajClip'].apply(lambda x: shapely.wkt.loads(x))
    gdf = gpd.GeoDataFrame(df, geometry='geom')
    return gdf

In [ ]:
query = """
SELECT ST_AsText(trajectory) AS traj FROM trips;
"""

all_trips_df = connection.execute(query).df()
all_trips_df['geom'] = all_trips_df['traj'].apply(lambda x: shapely.wkt.loads(x))
all_trips_gdf = gpd.GeoDataFrame(all_trips_df, geometry='geom')
all_trips_gdf['color'] = get_trip_colors(all_trips_gdf.shape[0])
all_trips_gdf.head()

In [ ]:
query2 = """
select municipalityid, municipalityname,
  COUNT(*) as intersect_count
from trips t, municipalities m
where eintersects(t.trip, m.municipalitygeo)
group by municipalityid, municipalityname
order by intersect_count desc;
"""

district_count_df = connection.execute(query2).df()
print(district_count_df.shape)
district_count_df.head()

In [ ]:
# Query 6.27 Fig. 6.23
num_rows = 2
num_cols = 3
graph_unit = 7

fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * graph_unit, num_rows * graph_unit), dpi=150)

#hanoi_gdf.plot(color=hanoi_gdf["color"], ax=axes[0][0], alpha=0.6)
#all_trips_gdf.plot(ax=axes[0][0], color=all_trips_gdf['color'], linewidth=1)
#axes[0][0].set_title("Districts of Hanoi")

for i in range(num_rows):
    for j in range(num_cols):
        idx = i * num_cols + j #- 1
        if idx >= 0:
            ax = axes[i][j]
            district_name = district_count_df.iloc[idx]['municipalityname']
            trip_count = district_count_df.iloc[idx]['intersect_count']
            district_geom_gdf = hanoi_gdf[hanoi_gdf['municipalityname'] == district_name]
            district_geom_gdf.plot(ax=ax, color=district_geom_gdf["color"], alpha=0.6)
            district_trips_gdf = get_district_trips(district_name)
            district_trips_gdf['color'] = get_trip_colors(district_trips_gdf.shape[0])
            district_trips_gdf.plot(ax=ax, color=district_trips_gdf['color'], linewidth=1)
            ax.set_title(f'{district_name}\n{trip_count} trips')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_rows = 2
num_cols = 3
graph_unit = 7

fig, axes = plt.subplots(
    num_rows, num_cols,
    figsize=(num_cols * graph_unit, num_rows * graph_unit),
    dpi=150
)

def square_bounds(minx, miny, maxx, maxy):
    """Expand bbox to a square so maps look proportional."""
    width = maxx - minx
    height = maxy - miny
    side = max(width, height)

    # Center the square
    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2

    half = side / 2
    return cx - half, cy - half, cx + half, cy + half

for i in range(num_rows):
    for j in range(num_cols):
        idx = i * num_cols + j
        ax = axes[i][j]

        district_name = district_count_df.iloc[idx]['municipalityname']
        trip_count = district_count_df.iloc[idx]['intersect_count']

        district_geom_gdf = hanoi_gdf[hanoi_gdf['municipalityname'] == district_name]

        # Plot geometry
        district_geom_gdf.plot(ax=ax, color=district_geom_gdf["color"], alpha=0.6)

        # Plot trips
        district_trips_gdf = get_district_trips(district_name)
        district_trips_gdf['color'] = get_trip_colors(district_trips_gdf.shape[0])
        district_trips_gdf.plot(ax=ax, color=district_trips_gdf['color'], linewidth=1)

        # --- compute proportional square bounds for THIS district ---
        minx, miny, maxx, maxy = district_geom_gdf.total_bounds
        minx, miny, maxx, maxy = square_bounds(minx, miny, maxx, maxy)

        ax.set_xlim(minx, maxx)
        ax.set_ylim(miny, maxy)
        ax.set_aspect('equal')

        ax.set_xticks([])
        ax.set_yticks([])

        ax.set_title(f'{district_name}\n{trip_count} trips')

plt.tight_layout()
plt.show()

In [ ]:
# Query 6.28. Total distance traveled per minicipality.
query = """
SELECT m.MunicipalityName,
  round((SUM(length(atGeometry(t.trip, m.MunicipalityGeo))) / 1000)::numeric, 3)
    AS TotalKm
FROM Trips t, Municipalities m
WHERE eintersects(t.trip, m.MunicipalityGeo)
GROUP BY m.MunicipalityName;"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

Execution time: 63.299775 seconds


,municipalityname,TotalKm
0,Huyện Ứng Hòa,1308.486
1,Huyện Mỹ Đức,1004.838
2,Huyện Quốc Oai,2197.561
3,Huyện Chương Mỹ,6121.844
4,Huyện Phú Xuyên,1468.059


In [36]:
query = "SELECT municipalityid, municipalityname, population, popdensitykm2, st_astext(municipalitygeo) as municipalitygeo  FROM municipalities;"  
hanoi = connection.execute(query).df()
hanoi.head(2)
hanoi['geom_obj'] = hanoi['municipalitygeo'].apply(lambda x: shapely.wkt.loads(x))
hanoi_gdf = gpd.GeoDataFrame(hanoi, geometry='geom_obj')
hanoi_gdf['color'] = sns.color_palette('hls', hanoi_gdf.shape[0]).as_hex()
hanoi_gdf.head(2)

,municipalityid,municipalityname,population,popdensitykm2,municipalitygeo,geom_obj,color
0,30,Huyện Ứng Hòa,215900,1147,"POLYGON ((11766093.96149793 2361530.927014738,...","POLYGON ((11766093.961 2361530.927, 11766147.6...",#db5f57
1,21,Huyện Mỹ Đức,210200,929,POLYGON ((11760645.796375375 2365247.918583854...,"POLYGON ((11760645.796 2365247.919, 11760834.0...",#db7957


In [ ]:
query = """
SELECT m.MunicipalityName,
  round((SUM(length(atGeometry(t.trip, m.MunicipalityGeo))) / 1000)::numeric, 3)
    AS TotalKm
FROM Trips t, Municipalities m
WHERE eintersects(t.trip, m.MunicipalityGeo)
GROUP BY m.MunicipalityName;"""

district_km_df = connection.execute(query).df()
district_km_df.head()

,municipalityname,TotalKm
0,Huyện Ứng Hòa,1308.486
1,Huyện Mỹ Đức,1004.838
2,Huyện Quốc Oai,2197.561
3,Huyện Chương Mỹ,6121.844
4,Huyện Phú Xuyên,1468.059


In [86]:
district_km_gdf = hanoi_gdf.merge(district_km_df, left_on='municipalityname', right_on='municipalityname', how='left')
district_km_gdf.head(2)

,municipalityid,municipalityname,population,popdensitykm2,municipalitygeo,geom_obj,color,TotalKm
0,30,Huyện Ứng Hòa,215900,1147,"POLYGON ((11766093.96149793 2361530.927014738,...","POLYGON ((11766093.961 2361530.927, 11766147.6...",#db5f57,1308.486
1,21,Huyện Mỹ Đức,210200,929,POLYGON ((11760645.796375375 2365247.918583854...,"POLYGON ((11760645.796 2365247.919, 11760834.0...",#db7957,1004.838


In [ ]:
_, axes = plt.subplots(figsize=(5, 5), dpi=150)
district_km_gdf.plot(column='TotalKm', ax=axes, legend=True, cmap='Oranges',
                     missing_kwds={
                        "color": "lightgrey",
                        "edgecolor": "whitesmoke"
                    })
axes.set_title('Total kilometers traveled in each district of Hanoi')
plt.show()

In [ ]:
# Query 6.28 Fig. 6.24 
 
# -----------------------------
# Ensure geometry is Shapely
# -----------------------------
if district_km_gdf.geom_obj.dtype == 'O':
    district_km_gdf['geom_obj'] = district_km_gdf['geom_obj'].apply(wkt.loads)
district_km_gdf = district_km_gdf.set_geometry('geom_obj')

# -----------------------------
#  Convert CRS to EPSG:4326
# -----------------------------
district_km_gdf_4326 = district_km_gdf.to_crs(4326)

# -----------------------------
# Create Folium map
# -----------------------------
center = district_km_gdf_4326.geometry.union_all().centroid
#m = folium.Map(location=[center.y, center.x], zoom_start=11, tiles='CartoDB positron')

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=11,
    min_zoom=5,
    max_zoom=18,
    zoom_control=True,
    tiles='CartoDB positron',
    control_scale=True
)

# Access the underlying Leaflet map to tweak zoomSnap
m.get_root().html.add_child(folium.Element("""
<script>
    var map = document.getElementsByClassName('folium-map')[0]._leaflet_map;
    map.options.zoomSnap = 0.05;  // allow quarter-step zoom
    map.options.zoomDelta = 0.05; // how much zoom changes per click/scroll
</script>
"""))

# -----------------------------
# Color scale from 0 to 10000
# -----------------------------
colormap = linear.Oranges_09.scale(0, 10000).to_step(n=10)

# Helper: convert RGBA to hex
def rgba_to_hex(rgba):
    r, g, b = [int(255*x) for x in rgba[:3]]
    return f"#{r:02x}{g:02x}{b:02x}"

# Colors for legend: light at bottom, dark at top
colors = [rgba_to_hex(c) if isinstance(c, tuple) else c for c in colormap.colors]  # DO NOT reverse
background = ', '.join(colors)

# -----------------------------
#  Only min/max labels fixed
# -----------------------------
ticks_html = f"""
<div style="display:flex; flex-direction:column; justify-content:space-between; height:150px; margin-right:5px;">
    <div style="text-align:right; font-size:12px;">10000</div>
    <div style="text-align:right; font-size:12px;">0</div>
</div>
"""

# -----------------------------
# Vertical legend with shorter bar
# -----------------------------
legend_html = f"""
<div style="
    position: fixed;
    top: 10px;
    right: 10px;
    width: 90px;  /* extra space for labels */
    height: 170px;  /* adjust to bar height + label spacing */
    z-index:9999;
    font-weight: regular;
">
  <div style="margin-bottom:5px; text-align:center; font-size:14px;">Total kilometers traveled</div>
  <div style="display:flex; flex-direction:row; height:150px; align-items:center;">
    {ticks_html}
    <div style="
        width: 20px;
        height: 150px;
        background: linear-gradient(to top, {background});
        border:1px solid black;
    "></div>
  </div>
</div>
"""

# Add legend to map
m.get_root().html.add_child(folium.Element(legend_html))

# -----------------------------
#  Add districts with proper coloring
# -----------------------------
for _, row in district_km_gdf_4326.iterrows():
    km = row['TotalKm']
    km_clipped = max(0, min(km, 10000))  # ensure within 0-10000
    color = colormap(km_clipped) if km_clipped is not None else "lightgrey"

    feature = {
        "type": "Feature",
        "geometry": mapping(row['geom_obj']),
        "properties": {"TotalKm": km}
    }

    folium.GeoJson(
        feature,
        style_function=lambda feature, col=color: {
            "fillColor": col,
            "color": "whitesmoke",
            "weight": 0.8,
            "fillOpacity": 0.7
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["TotalKm"],
            aliases=["Total Km:"],
            localize=True
        )
    ).add_to(m)

# -----------------------------
# Show map
# -----------------------------
m

In [ ]:
# Query 6.29. Pairs of vehicles that have ever been at less than ten meters from each other.
query = """
SELECT DISTINCT t1.VehicleId AS VehicleId1, t1.TripId AS TripId1, 
  ST_AsText(t1.Trajectory) AS Traj1, t2.VehicleId AS VehicleId2, t2.TripId AS TripId2,
  ST_AsText(t2.Trajectory) AS Traj2
FROM (SELECT * FROM trips t1 ORDER BY TripId LIMIT 1000) t1,
     (SELECT * FROM trips t2 ORDER BY TripId LIMIT 1000) t2
WHERE t1.VehicleId < t2.VehicleId AND eDwithin(t1.Trip, t2.Trip, 10.0)
ORDER BY t1.VehicleId, t2.VehicleId;"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

Execution time: 86.845326 seconds


,VehicleId1,TripId1,Traj1,VehicleId2,TripId2,Traj2
0,1,1,LINESTRING (11781545.407382663 2393706.0958389...,5,56,LINESTRING (11738957.509039713 2406010.7155046...
1,1,10,LINESTRING (11781545.407382663 2393706.0958389...,5,63,LINESTRING (11738957.509039713 2406010.7155046...
2,1,11,LINESTRING (11733388.929623965 2421155.8332119...,8,110,LINESTRING (11797701.060366152 2375476.4775091...
3,1,13,LINESTRING (11733388.929623965 2421155.8332119...,8,114,LINESTRING (11797701.060366152 2375476.4775091...
4,1,12,LINESTRING (11781545.407382663 2393706.0958389...,44,637,LINESTRING (11780474.703124367 2392172.4675870...


In [ ]:
query = """
SELECT DISTINCT t1.VehicleId AS VehicleId1, t1.TripId AS TripId1, 
  ST_AsText(t1.Trajectory) AS Traj1, t2.VehicleId AS VehicleId2, t2.TripId AS TripId2,
  ST_AsText(t2.Trajectory) AS Traj2
FROM (SELECT * FROM trips t1 LIMIT 100) t1,
     (SELECT * FROM trips t2 LIMIT 100) t2
WHERE t1.VehicleId < t2.VehicleId AND eDwithin(t1.Trip, t2.Trip, 10.0)
ORDER BY t1.VehicleId, t2.VehicleId LIMIT 1; 
"""
vehicle_pair_df = connection.execute(query).df()
vehicle_pair_df

vehicle1_df = vehicle_pair_df[['VehicleId1', 'TripId1', 'Traj1']].copy()
vehicle2_df = vehicle_pair_df[['VehicleId2', 'TripId2', 'Traj2']].copy()

vehicle1_df['geom'] = vehicle1_df['Traj1'].apply(lambda x: shapely.wkt.loads(x))
vehicle1_gdf = gpd.GeoDataFrame(vehicle1_df, geometry='geom')

vehicle2_df['geom'] = vehicle2_df['Traj2'].apply(lambda x: shapely.wkt.loads(x))
vehicle2_gdf = gpd.GeoDataFrame(vehicle2_df, geometry='geom') 

In [ ]:
# ---------------------------------------------------------
#  Convert WKT to GeoDataFrames
# ---------------------------------------------------------
vehicle1_df = vehicle1_df.copy()
vehicle2_df = vehicle2_df.copy()

vehicle1_df['geom'] = vehicle1_df['Traj1'].apply(shapely.wkt.loads)
vehicle2_df['geom'] = vehicle2_df['Traj2'].apply(shapely.wkt.loads)

vehicle1_gdf = gpd.GeoDataFrame(vehicle1_df, geometry='geom')
vehicle2_gdf = gpd.GeoDataFrame(vehicle2_df, geometry='geom')

# ---------------------------------------------------------
#  Set CRS = 3857 (your SRID) and convert to 4326
# ---------------------------------------------------------
vehicle1_gdf = vehicle1_gdf.set_crs(3857, allow_override=True).to_crs(4326)
vehicle2_gdf = vehicle2_gdf.set_crs(3857, allow_override=True).to_crs(4326)

# ---------------------------------------------------------
#  Calculate map center
# ---------------------------------------------------------
merged = pd.concat([vehicle1_gdf, vehicle2_gdf], ignore_index=True)
merged_gdf = gpd.GeoDataFrame(merged, geometry='geom')
center = merged_gdf.geometry.union_all().centroid

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=13,
    tiles="CartoDB positron"
)

# ---------------------------------------------------------
#  Add Vehicle 1 trajectory (BLUE)
# ---------------------------------------------------------
for _, row in vehicle1_gdf.iterrows():
    folium.GeoJson(
        row['geom'].__geo_interface__,
        style_function=lambda x: {"color": "blue", "weight": 4, "opacity": 0.9},
        tooltip=f"Vehicle {row['VehicleId1']}<br>Trip {row['TripId1']}"
    ).add_to(m)

# ---------------------------------------------------------
# Add Vehicle 2 trajectory (RED)
# ---------------------------------------------------------
for _, row in vehicle2_gdf.iterrows():
    folium.GeoJson(
        row['geom'].__geo_interface__,
        style_function=lambda x: {"color": "red", "weight": 2, "opacity": 0.7},
        tooltip=f"Vehicle {row['VehicleId2']}<br>Trip {row['TripId2']}"
    ).add_to(m)

# ---------------------------------------------------------
#  Add Legend
# ---------------------------------------------------------
 
legend_html = """
<div style="
    position: fixed;
    bottom: 40px; left: 40px;
    z-index: 9999; font-size: 12px; font-weight: normal; font-family: Arial, sans-serif;
    background: white; padding: 10px;
   # border: 2px solid #555; border-radius: 6px;
">
<b>Trajectories</b><br>
<span style="color: blue;">&#9632;</span> Vehicle 1<br>
<span style="color: red;">&#9632;</span> Vehicle 2
</div>
"""
 
m.get_root().html.add_child(folium.Element(legend_html))

# ---------------------------------------------------------
# Display Map
# ---------------------------------------------------------
m

In [ ]:
query1 = """
CREATE OR REPLACE TABLE  Licences 
(LicenceId integer, licence text, VehicleId integer);
"""
connection.execute(query1)

In [ ]:
conn_str = PG_HANOI

query = f"""
INSERT INTO Licences 
SELECT * FROM postgres_scan(
    '{conn_str}',    
    'public',          
    'licences'       
);
"""
result = connection.execute(query).fetchdf()

In [ ]:
answer = connection.execute("SELECT * FROM LICENCES").df()
answer.head()

In [86]:
query2 = """
CREATE OR REPLACE TABLE  Vehicles
(vehicleid integer, licence text, VehicleType text, model text);
"""
connection.execute(query2)

In [ ]:
connection.execute("CREATE OR REPLACE TABLE  Licences1 (LicenceId integer, licence text, VehicleId integer);")

In [ ]:
connection.execute("INSERT INTO Licences1 (SELECT * FROM LICENCES LIMIT 10);")

In [11]:
answer = connection.execute("SELECT * FROM LICENCES1").df()
answer.head()

,LicenceId,licence,VehicleId
0,17,B-CK 4,4
1,98,B-QZ 5,5
2,99,B-ZY 6,6
3,1,B-ZY 6,6
4,48,B-OM 9,9


In [ ]:
query3 = f"""
INSERT INTO Vehicles
SELECT * FROM postgres_scan(
    '{conn_str}',    
    'public',          
    'vehicles'       
);
"""
result2 = connection.execute(query3).fetchdf()

In [88]:
answer = connection.execute("SELECT * FROM VEHICLES").df()
answer.head()

,vehicleid,licence,vehtype,model
0,1,B-EF 1,passenger,Sachsenring
1,2,B-OU 2,passenger,Multicar
2,3,B-UM 3,passenger,Acabion
3,4,B-CK 4,passenger,Multicar
4,5,B-QZ 5,passenger,Acabion


In [ ]:
# Query 6.30

#For each trip and vehicle, list the points from table Points 
#that have the vehicle corresponding to the trip, among their three 
#nearest neighbors. List also the distance between the point and the vehicle. 
#For simplicity, you can use a reduced part of the Trips (say, the first 100 trips). 
query = """
WITH TripsTraj AS(
 SELECT TripId, VehicleId, trajectory
 FROM Trips),
 
PointTrips AS(
 SELECT P.PointId, T2.VehicleId, T2.TripId, T2.Distance
 FROM Points P,
	  LATERAL
        (SELECT T1.VehicleId, T1.TripId, 
		        st_distance(P.geom,T1.Trajectory) AS Distance
        FROM TripsTraj T1
        ORDER BY Distance LIMIT 3) AS T2
)

SELECT T.TripId, T.VehicleId, P.PointId, PT.Distance
FROM Trips100 T  JOIN PointTrips PT ON
     (T.VehicleId = PT.VehicleId  AND T.TripId = PT.TripId)
	 JOIN Points P ON P.PointId = PT.PointId
ORDER BY T.VehicleId, T.TripId, P.PointId;
"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

In [ ]:
# Query 6.31
# Also consider 30-minutes intervals between 2020-06-01  00:00:00 and 2020-06-02  23:30:00.   
#How many vehicles were active per comune and per interval?  

query = """ DROP TABLE IF EXISTS Bins;
CREATE TABLE Bins(PId,SpanP) AS
WITH Intervals AS
(SELECT 
    ROW_NUMBER() OVER() AS pid,
    h AS startt,
    h + INTERVAL '30 minutes' AS endt
FROM (
    SELECT generate_series AS h
    FROM generate_series(
        TIMESTAMPTZ '2020-06-01 00:00:00',
        TIMESTAMPTZ '2020-06-02 23:30:00',
        INTERVAL '30 minutes'
    )
) t )

SELECT Pid, span(startt::timestamptz,endt::timestamptz) as SpanP
FROM Intervals;
"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

In [ ]:
# Also consider 30-minutes intervals between 2020-06-01  00:00:00 and 2020-06-02  23:30:00.   
#How many vehicles were active per comune and per interval?  

query = """ WITH TripPeriod AS(
 SELECT tripid, pId, p.SpanP, vehicleid, atTime(T.Trip, P.SpanP) as tripPeriod
 FROM Trips T, Bins P
 WHERE T.Trip && P.SpanP
)
SELECT T.Pid as intervalId, T.SpanP as intervalTime, M.municipalityName as Municipality, COUNT(*)  
        
FROM TripPeriod T, Municipalities M 
WHERE   eintersects(tripPeriod, M.MunicipalityGeo)
GROUP BY T.Pid,T.SpanP,  M.municipalityName
ORDER BY COUNT(*) desc, M.municipalityName, T.Pid;
"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

In [ ]:
# Query 6.32. When and where did the vehicles with license plate numbers
#from  Licenses1 meet other vehicles (distance < 3 meters) and what are the latter's licenses?
query = """
WITH Meetings AS (
  SELECT l1.Licence AS Licence1,
    t2.VehicleId AS Car2Id,  whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0)) AS Periods,
    atTime(t1.trip, whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0))) as tt1,
    atTime(t2.trip, whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0))) as tt2
  FROM Trips t1, Licences1 l1, Trips t2, Vehicles v
  WHERE t1.VehicleId = l1.VehicleId AND
    t2.VehicleId = v.VehicleId AND
    t1.VehicleId <> t2.VehicleId AND
    t2.Trip && expandSpace(t1.trip::STBOX, 3.0) )
SELECT Licence1, Car2Id, st_astext(trajectory(tt1)) AS geom1, 
st_astext(trajectory(tt2)) AS geom2, Periods::VARCHAR
FROM Meetings
WHERE Periods IS NOT NULL;"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

Execution time: 30.105590 seconds


,Licence1,Car2Id,geom1,geom2,CAST(Periods AS VARCHAR)
0,B-CK 4,3,LINESTRING (11771279.811704397 2385417.0716443...,LINESTRING (11771287.820033683 2385423.0605801...,"{[2020-06-02 14:17:00.714409-03, 2020-06-02 14..."
1,B-CK 4,194,LINESTRING (11771611.343818286 2385695.9297957...,LINESTRING (11771617.855627248 2385703.5190255...,"{[2020-06-02 14:18:00.014616-03, 2020-06-02 14..."
2,B-CK 4,194,LINESTRING (11769683.77896123 2384426.82440550...,LINESTRING (11769674.968153924 2384422.0946699...,"{[2020-06-03 07:22:38.039694-03, 2020-06-03 07..."
3,B-CK 4,129,LINESTRING (11777190.919879809 2390265.9827639...,LINESTRING (11777195.361141032 2390274.9424091...,"{[2020-06-03 15:00:50.443759-03, 2020-06-03 15..."
4,B-CK 4,190,LINESTRING (11768407.010787118 2383641.6566108...,LINESTRING (11768398.8560936 2383635.868598232...,"{[2020-06-04 06:24:54.896961-03, 2020-06-04 06..."


In [ ]:
# Query 6.32. Figure 6.27

# -----------------------------------------
# RUN QUERY
# -----------------------------------------
query = """
WITH Meetings AS (
  SELECT l1.Licence AS Licence1,
    t2.VehicleId AS Car2Id,
    whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0)) AS Periods,
    atTime(t1.trip, whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0))) as tt1,
    atTime(t2.trip, whenTrue(tDwithin(t1.Trip, t2.Trip, 3.0))) as tt2
  FROM Trips t1, Licences1 l1, Trips t2, Vehicles v
  WHERE t1.VehicleId = l1.VehicleId
    AND t2.VehicleId = v.VehicleId
    AND t1.VehicleId <> t2.VehicleId
    AND t2.Trip && expandSpace(t1.trip::STBOX, 3.0)
)
SELECT Licence1,
       Car2Id,
       st_astext(trajectory(tt1)) AS geom1,
       st_astext(trajectory(tt2)) AS geom2,
       Periods::VARCHAR
FROM Meetings
WHERE Periods IS NOT NULL;
"""

In [26]:
df.head()

,Licence1,Car2Id,geom1,geom2,CAST(Periods AS VARCHAR)
0,B-CK 4,3,LINESTRING (11771279.811704397 2385417.0716443...,LINESTRING (11771287.820033683 2385423.0605801...,"{[2020-06-02 14:17:00.714409-03, 2020-06-02 14..."
1,B-CK 4,194,LINESTRING (11771611.343818286 2385695.9297957...,LINESTRING (11771617.855627248 2385703.5190255...,"{[2020-06-02 14:18:00.014616-03, 2020-06-02 14..."
2,B-CK 4,194,LINESTRING (11769683.77896123 2384426.82440550...,LINESTRING (11769674.968153924 2384422.0946699...,"{[2020-06-03 07:22:38.039694-03, 2020-06-03 07..."
3,B-CK 4,129,LINESTRING (11777190.919879809 2390265.9827639...,LINESTRING (11777195.361141032 2390274.9424091...,"{[2020-06-03 15:00:50.443759-03, 2020-06-03 15..."
4,B-CK 4,190,LINESTRING (11768407.010787118 2383641.6566108...,LINESTRING (11768398.8560936 2383635.868598232...,"{[2020-06-04 06:24:54.896961-03, 2020-06-04 06..."


In [ ]:
# --------------------------------------------------------
# 1) Convert strings → geometry if needed
# --------------------------------------------------------
def ensure_geom(x):
    if isinstance(x, str):
        return shapely.wkt.loads(x)
    return x

df["geom1"] = df["geom1"].apply(ensure_geom)
df["geom2"] = df["geom2"].apply(ensure_geom)

# --------------------------------------------------------
# 2) Normalize all geometry types into LineStrings
# --------------------------------------------------------
def normalize_geom(g):
    if g is None:
        return []

    if isinstance(g, LineString):
        return [g]

    if isinstance(g, MultiLineString):
        return list(g.geoms)

    if isinstance(g, Point):
        return [LineString([g, g])]

    if isinstance(g, MultiPoint):
        return [LineString([pt, pt]) for pt in g.geoms]

    if isinstance(g, GeometryCollection):
        out = []
        for sub in g.geoms:
            out.extend(normalize_geom(sub))
        return out

    return []

# --------------------------------------------------------
# 3) Convert into GeoDataFrames (3857 → 4326)
# --------------------------------------------------------
gdf1 = gpd.GeoDataFrame(df.copy(), geometry=df["geom1"], crs=3857)
gdf2 = gpd.GeoDataFrame(df.copy(), geometry=df["geom2"], crs=3857)

gdf1_4326 = gdf1.to_crs(4326)
gdf2_4326 = gdf2.to_crs(4326)

# --------------------------------------------------------
# 4) Map center = centroid of first geometry (already 4326)
# --------------------------------------------------------
first_geom = gdf1_4326.geometry.iloc[0]

# guarantee it's a Point for centering
if isinstance(first_geom, (LineString, MultiLineString)):
    cy, cx = first_geom.coords[0][1], first_geom.coords[0][0]
else:
    cy, cx = 0, 0  # fallback but won't be needed

m = folium.Map(location=[cy, cx], zoom_start=13, tiles="CartoDB positron")

# --------------------------------------------------------
# 5) Draw geometries
# --------------------------------------------------------
for idx in range(len(df)):
    geom1 = gdf1_4326.geometry.iloc[idx]
    geom2 = gdf2_4326.geometry.iloc[idx]

    parts1 = normalize_geom(geom1)
    parts2 = normalize_geom(geom2)

    # --- DEBUG (remove if you want) ---
    print(f"Row {idx}: geom1 parts={len(parts1)}, geom2 parts={len(parts2)}")

    # --- GEOM 1 (BLUE) ---
    for seg in parts1:
        folium.PolyLine(
            [(lat, lon) for lon, lat in seg.coords],
            color="#0066FF",         # strong blue
            weight=5,
            opacity=1.0
        ).add_to(m)

    # --- GEOM 2 (RED) ---
    for seg in parts2:
        folium.PolyLine(
            [(lat, lon) for lon, lat in seg.coords],
            color="#FF0000",         # strong red
            weight=5,
            opacity=1.0
        ).add_to(m)

m

In [ ]:
#Query 6.33. For each trip from Trips, list the three vehicles that have been  closest to that vehicle, together with the distance. 
#For simplicity, you can use a reduced part of the Trips (say, the first 100 trips). 
query = """
CREATE OR REPLACE VIEW Trips200 AS (SELECT * FROM trips LIMIT 10);

SELECT T1.tripid, T1.VehicleId AS VehId1, C2.VehicleId AS VehId2, C2.Distance
FROM Trips200 T1,
     LATERAL
     (SELECT T2.VehicleId, minValue(T1.Trip <-> T2.Trip) AS Distance
      FROM Trips200 T2
      WHERE T1.VehicleId < T2.VehicleId AND 
	        TimeSpan(T1.Trip) && TimeSpan(T2.Trip)
     ORDER BY Distance LIMIT 3) AS C2
ORDER BY T1.tripid, T1.VehicleId, C2.Distance, C2.VehicleId;
"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

In [ ]:
#QUERY 6.34. For each group of ten disjoint vehicles, list the point(s) from Points, having the minimum aggregated distance 
#from the given group of ten vehicles during the given period.   
#CREATE OR REPLACE VIEW Trips100 AS (SELECT * FROM trips LIMIT 100);

query = """
WITH Groups AS
(
 SELECT ((ROW_NUMBER() OVER 
		 (ORDER BY V.Vehicleid))-1)/10 + 1 AS GroupId, V.Vehicleid
 FROM Vehicles V
),
SumDistances AS (
 SELECT G.GroupId, P.PointId, 
	    SUM(ST_Distance(T.trajectory, P.Geom)) AS SumDist
 FROM  Groups G, Points P, Trips T
 WHERE T.Vehicleid = G.Vehicleid
 GROUP BY G.GroupId, P.PointId)
 
SELECT S1.GroupId, S1.PointId, S1.SumDist
FROM SumDistances S1 
WHERE S1.SumDist <= ALL
  (SELECT SumDist 
   FROM SumDistances S2 
   WHERE S1.GroupId = S2.GroupId )
ORDER BY S1.GroupId, S1.PointId;
"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

In [ ]:
conn_str = PG_HANOI

query12 = f"""
INSERT INTO Points
SELECT * FROM postgres_scan(
    '{conn_str}',    
    'public',          
    'points'       
);
"""
result2 = connection.execute(query12).fetchdf()

In [ ]:
query1 = """CREATE OR REPLACE VIEW Points1 AS (SELECT * FROM Points LIMIT 10); 
"""
connection.execute(query)
query2 = """CREATE OR REPLACE VIEW Regions1 AS (SELECT * FROM Regions LIMIT 10); 
"""
connection.execute(query)

In [22]:
# Query 6.35. For each passenger vehicle trip, find all points such that the trip
#passes within 10 meters of them. Additionally, for each such point, return the
#trip, the vehicle, the point, the trip trajectory, and the exact time when the
#vehicle first reaches that point.

#query = """WITH Timestamps AS (
query = """
  SELECT DISTINCT t.TripId, v.Licence, p.PointId, st_astext(p.Geom),
    startTimestamp(atValue(t.Trip, p.Geom))  AS Instant,
    ST_AsText(t.trajectory) as trajectory
  FROM Trips t, Vehicles v, Points p
  WHERE t.VehicleId = v.VehicleId AND
    v.VehicleType = 'passenger' AND
    eIntersects(t.Trip, ST_Buffer(p.Geom,10))
 ;
"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

Execution time: 77.372307 seconds


,tripid,licence,pointid,st_astext(p.Geom),Instant,trajectory
0,426,B-FT 29,26,POINT (11751305.656951806 2431639.985166498),2020-06-05 05:10:45.997204-03:00,LINESTRING (11751001.966248972 2433121.7680177...
1,427,B-FT 29,26,POINT (11751305.656951806 2431639.985166498),2020-06-05 16:00:53.248481-03:00,LINESTRING (11795085.55327022 2430852.97079237...
2,414,B-FT 29,26,POINT (11751305.656951806 2431639.985166498),2020-06-01 05:28:15.374502-03:00,LINESTRING (11751001.966248972 2433121.7680177...
3,415,B-FT 29,26,POINT (11751305.656951806 2431639.985166498),2020-06-01 15:33:57.711446-03:00,LINESTRING (11795085.55327022 2430852.97079237...
4,416,B-FT 29,26,POINT (11751305.656951806 2431639.985166498),2020-06-01 17:35:40.293322-03:00,LINESTRING (11751001.966248972 2433121.7680177...


In [ ]:
# ============================================================
# 1) Convert WKT → shapely
# ============================================================

df["point_geom"] = df["st_astext(p.Geom)"].apply(shapely.wkt.loads)
df["traj_geom"]  = df["trajectory"].apply(shapely.wkt.loads)

# ============================================================
# 2) Build GeoDataFrames
# ============================================================

gdf_points = gpd.GeoDataFrame(df, geometry="point_geom", crs="EPSG:3857")
gdf_traj   = gpd.GeoDataFrame(df, geometry="traj_geom",  crs="EPSG:3857")

# Convert to 4326
gdf_points_4326 = gdf_points.to_crs(4326)
gdf_traj_4326   = gdf_traj.to_crs(4326)

# ============================================================
# 3) Helper → extract list of line segments for ANY geometry
# ============================================================

def extract_lines(geom):
    """Returns a list of lists of (lat, lon) pairs"""
    
    if isinstance(geom, LineString):
        return [[(y, x) for x, y in geom.coords]]
    
    if isinstance(geom, MultiLineString):
        return [[(y, x) for x, y in line.coords] for line in geom]
    
    if isinstance(geom, (Point, MultiPoint)):
        return []
    
    if isinstance(geom, GeometryCollection):
        lines = []
        for g in geom:
            lines.extend(extract_lines(g))
        return lines

    return []


# ============================================================
# 4) Build folium map
# ============================================================

center = gdf_traj_4326.geometry.union_all().centroid
m = folium.Map(location=[center.y, center.x], zoom_start=12, tiles="CartoDB positron")

# ============================================================
# 5) Add BLUE trajectories
# ============================================================

for _, row in gdf_traj_4326.iterrows():
    geom = row["traj_geom"]          # ← FIXED
    segments = extract_lines(geom)
    
    for seg in segments:
        folium.PolyLine(
            locations=seg,
            color="blue",
            weight=3,
            opacity=0.9,
            tooltip=f"Trip {row['tripid']} | {row['licence']}"
        ).add_to(m)

# ============================================================
# 6) Add RED points
# ============================================================

for _, row in gdf_points_4326.iterrows():
    pt = row["point_geom"]           # ← FIXED
    folium.CircleMarker(
        location=(pt.y, pt.x),
        radius=4,
        color="red",
        fill=True,
        fill_opacity=0.9,
        tooltip=f"Point {row['pointid']} | {row['Instant']}"
    ).add_to(m)

m

In [31]:
# Query 6.36. Trips that pass through a point in table points, and the first instant when this happenned
#query = """WITH Timestamps AS (
query = """
WITH Timestamps AS (SELECT DISTINCT  v.Licence, p.pointId, p.Geom,
    MIN(startTimestamp(atValue(t.Trip, p.Geom))) AS Instant
  FROM Trips t, Vehicles v, Points p
  WHERE t.VehicleId = v.VehicleId AND
    v.VehicleType = 'passenger' AND
    eIntersects(t.Trip, p.Geom)
  GROUP BY v.Licence, p.pointId, p.Geom)
SELECT t1.*
FROM Timestamps t1
WHERE t1.Instant <= ALL (
  SELECT t2.Instant
  FROM Timestamps t2
  WHERE t1.PointId = t2.PointId )
ORDER BY t1.PointId, t1.Licence;
"""
t0 = time.perf_counter()
answer = connection.execute(query).df()
t1 = time.perf_counter()
print(f"Execution time: {t1 - t0:.6f} seconds")
answer.head()

Execution time: 76.451387 seconds


,licence,pointid,geom,Instant
0,B-FT 29,26,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2020-06-01 05:28:15.374502-03:00
1,B-NU 187,47,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2020-06-01 06:05:49.000881-03:00
2,B-CR 108,70,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2020-06-01 05:53:57.720774-03:00
3,B-DB 20,77,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2020-06-04 21:14:27.402708-03:00
4,B-KH 14,83,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...",2020-06-02 20:10:02.536401-03:00
